In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../..')

In [ ]:
from Combined.DataPrep.Data_Prep import Combined_Data_Prep
from Pro.DataPrep import Prep_Map, Output_Map
from College.DataPrep import Prep_Map as C_Prep_Map, Output_Map as C_Output_Map

data_prep = Combined_Data_Prep(
    Prep_Map.base_prep_map, 
    Output_Map.base_output_map,
    C_Prep_Map.college_base_prep_map,
    C_Output_Map.college_output_map)

hitter_io_list = data_prep.Generate_IO_Hitters(is_training=True)

In [ ]:
from Combined.DataPrep.Player_Dataset import Create_Test_Train_Datasets

train_dataset, test_dataset = Create_Test_Train_Datasets(
    player_list=hitter_io_list, 
    is_hitter=True)

In [ ]:
from Pro.Model.Player_Model import RNN_Model as Pro_Model
import torch
from Constants import device

pro_model = Pro_Model(
    input_size=train_dataset.GetProInputSize(),
    mutators=torch.empty(0),
    data_prep=data_prep.pro_data_prep,
    is_hitter=True,
).to(device)

In [ ]:
from Combined.Tuning.ColWar import objective
from functools import partial

objective_func = partial(
    objective,
    pro_network=pro_model,
    train_dataset=train_dataset,
    test_dataset=test_dataset,
    data_prep=data_prep.college_data_prep,
    is_hitter=True,
    num_repeats=1,
)


In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction="minimize",
    load_if_exists=True,
    study_name="Col_WAR_Tuning",
    storage="sqlite:///Col_Tune.db"
)

In [ ]:
study.optimize(objective_func, n_trials=300, show_progress_bar=True)

In [ ]:
print(f"Final best value after refinement: {study.best_value:.4f}")
print(f"Best hyperparameters: {study.best_params}")

In [ ]:
from tqdm import tqdm

num_top_trials = 20
num_repeats_refine = 3

top_trials = sorted(
    [t for t in study.trials if t.value is not None],
    key=lambda t: t.value
)[:num_top_trials]

best_value = float('inf')

for i, trial in enumerate(tqdm(top_trials)):
    # Create a new partial with higher repeats
    objective_func_refine = partial(
        objective,
        pro_network=pro_model,
        train_dataset=train_dataset,
        test_dataset=test_dataset,
        data_prep=data_prep.college_data_prep,
        is_hitter=True,
        num_repeats=num_repeats_refine,
    )
    
    # Create a new trial with the same hyperparameters
    new_trial = optuna.trial.FixedTrial(trial.params)
    
    refined_value = objective_func_refine(new_trial)
    # Update overall best if this is better
    if refined_value < best_value:
        best_value = refined_value
        best_params = trial.params



In [ ]:
print(f"Final best value after refinement: {best_value:.4f}")
print(f"Best hyperparameters: {best_params}")

In [ ]:
import optuna.visualization as vis
vis.plot_param_importances(study).show()

In [ ]:
vis.plot_optimization_history(study).show()

In [ ]:
vis.plot_slice(study).show()

In [ ]:
vis.plot_contour(study, params=["hidden_size", "dropout"]).show()

In [ ]:
vis.plot_parallel_coordinate(study).show()